In [1]:
! pip install scikit-learn

In [2]:
! pip install scikit-multilearn

In [ ]:
import ast
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, confusion_matrix

In [4]:
df_encoded = pd.read_csv("df_encoded.csv")

In [5]:
df_encoded_nan= pd.read_csv("df_encoded_nan.csv")

In [6]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106848 entries, 0 to 106847
Data columns (total 60 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   rating                  106848 non-null  int64  
 1   startYear               106848 non-null  int64  
 2   runtimeMinutes          87956 non-null   float64
 3   numVotes                106848 non-null  int64  
 4   totalCredits            106848 non-null  int64  
 5   criticReviewsTotal      106848 non-null  int64  
 6   numRegions              106848 non-null  int64  
 7   userReviewsTotal        106848 non-null  int64  
 8   companiesNumber         106848 non-null  int64  
 9   averageRating           106848 non-null  float64
 10  externalLinks           106848 non-null  int64  
 11  writerCredits           106848 non-null  int64  
 12  directorsCredits        106848 non-null  int64  
 13  quotesTotal             106848 non-null  int64  
 14  totalMedia          

# Classificazione Generale per la scelta dei modelli

In [7]:
# Divisione train/test
X = df_encoded.drop(columns= ['runtimeMinutes', "genres_Film-Noir", "from_Africa",
    "from_Asia",
    "from_Europe",
    "from_North America",
    "from_Oceania",
    "from_South America" ])

y = df_encoded[["from_Africa",
    "from_Asia",
    "from_Europe",
    "from_North America",
    "from_Oceania",
    "from_South America"]]

In [8]:
from sklearn.preprocessing import StandardScaler

features_to_scale = [
    'rating', 'startYear', 'numVotes', 'totalCredits',
    'criticReviewsTotal', 'numRegions', 'userReviewsTotal', 'companiesNumber',
    'averageRating', 'externalLinks', 'writerCredits', 'directorsCredits',
    'quotesTotal', 'totalMedia', 'totalNominations'
]

# Tutte le altre colonne (che non devono essere scalate)
features_not_scaled = [col for col in X.columns if col not in features_to_scale]


In [9]:
# Inizializza lo scaler
scaler = StandardScaler()

# Applica lo scaling solo sulle colonne selezionate
X_scaled_part = pd.DataFrame(
    scaler.fit_transform(X[features_to_scale]),
    columns=features_to_scale,
    index=X.index
)

# Prendi le altre colonne così come sono
X_not_scaled_part = X[features_not_scaled]


In [10]:
X_prepared = pd.concat([X_scaled_part, X_not_scaled_part], axis=1)

# Ordina le colonne per sicurezza (opzionale)
X_prepared = X_prepared[X.columns]  # per mantenere l'ordine originale


In [11]:
from skmultilearn.model_selection import iterative_train_test_split

# Converti in NumPy array
X_np = X_prepared.values
y_np = y.values

# Split stratificato multilabel
X_train_np, y_train_np, X_test_np, y_test_np = iterative_train_test_split(
    X_np, y_np, test_size=0.3
)



In [12]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, hamming_loss, accuracy_score
from sklearn.multiclass import OneVsRestClassifier

svm_clf = OneVsRestClassifier(SVC(probability=True))
rf_clf = OneVsRestClassifier(RandomForestClassifier(random_state=42))
lr_clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))

#svm_clf.fit(X_train_np, y_train_np)
rf_clf.fit(X_train_np, y_train_np)
#lr_clf.fit(X_train_np, y_train_np)

#svm_pred = svm_clf.predict(X_test_np)
rf_pred = rf_clf.predict(X_test_np)
#lr_pred = lr_clf.predict(X_test_np)

# Funzione di valutazione multilabel
def evaluate_model(y_true, y_pred, model_name):
    print(f"--- Risultati per {model_name} ---")
    print(f"Hamming Loss: {hamming_loss(y_true, y_pred):.4f}")
    print(f"Accuracy (subset accuracy): {accuracy_score(y_true, y_pred):.4f}")
    print("Classification Report (per classe):")
    print(classification_report(y_true, y_pred))
    print("\n")

# Valutazione
#evaluate_model(y_test_np, svm_pred, "SVM")
evaluate_model(y_test_np, rf_pred, "Random Forest")
#evaluate_model(y_test_np, lr_pred, "Logistic Regression")


--- Risultati per Random Forest ---
Hamming Loss: 0.0785
Accuracy (subset accuracy): 0.6683
Classification Report (per classe):
              precision    recall  f1-score   support

           0       0.94      0.10      0.18       309
           1       0.84      0.50      0.63      4799
           2       0.79      0.64      0.71     11156
           3       0.86      0.82      0.84     15583
           4       1.00      0.31      0.47       698
           5       0.96      0.04      0.07       739

   micro avg       0.84      0.68      0.75     33284
   macro avg       0.90      0.40      0.48     33284
weighted avg       0.84      0.68      0.73     33284
 samples avg       0.70      0.69      0.69     33284





/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [15]:
svm_clf.fit(X_train_np, y_train_np)
svm_pred = svm_clf.predict(X_test_np)
evaluate_model(y_test_np, svm_pred, "SVM")

--- Risultati per SVM ---
Hamming Loss: 0.0999
Accuracy (subset accuracy): 0.5617
Classification Report (per classe):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       309
           1       0.82      0.34      0.48      4799
           2       0.73      0.52      0.61     11156
           3       0.82      0.75      0.78     15583
           4       0.96      0.18      0.30       698
           5       0.00      0.00      0.00       739

   micro avg       0.79      0.58      0.67     33284
   macro avg       0.55      0.30      0.36     33284
weighted avg       0.77      0.58      0.64     33284
 samples avg       0.59      0.59      0.59     33284





/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier

In [16]:
import joblib
model_path = "/Applications/OneDrive.app"
joblib.dump(
    svm_clf,
    model_path + 'svm_continent_DM2'+ '.pkl'
)

['/Applications/OneDrive.appsvm_continent_DM2.pkl']

In [14]:
lr_clf.fit(X_train_np, y_train_np)
lr_pred = lr_clf.predict(X_test_np)
evaluate_model(y_test_np, lr_pred, "Logistic Regression")

--- Risultati per Logistic Regression ---
Hamming Loss: 0.1324
Accuracy (subset accuracy): 0.3981
Classification Report (per classe):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       309
           1       0.73      0.24      0.36      4799
           2       0.56      0.28      0.37     11156
           3       0.74      0.65      0.69     15583
           4       0.00      0.00      0.00       698
           5       0.00      0.00      0.00       739

   micro avg       0.69      0.43      0.53     33284
   macro avg       0.34      0.20      0.24     33284
weighted avg       0.64      0.43      0.50     33284
 samples avg       0.43      0.44      0.43     33284





/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier